# How 1-WL works inside Referential Alignment — a richer worked example

This is a companion to `explain_ra_wl.ipynb`. That first notebook used the
simplest possible setup — one id scope, a chain of *identical* nodes — so the
intermediate outputs were deliberately bare (an all-`1.0` cost matrix, empty WL
labels, no hub vertices, a one-line topological order).

Here we step up to a **two-scope** model so every internal output has something
interesting in it:

- a **mixed** property cost matrix (some people distinct, one pair tied),
- **non-empty WL labels** — both typed-edge attributes *and* a cross-scope fold,
- **hub vertices** from a k-ary "membership" relation,
- a **non-trivial topological order** between the two scopes,
- a **two-round** color refinement where a tie survives round 1 and breaks in
  round 2.

We also pause up front to nail down a term that is easy to find hazy: the
**"definer"**.

> As before, every code cell drives the *real* internal methods. Nothing is faked.


## 0. The one-sentence problem (recap)

When fields are marked as **ids** (`idScope`) and others as **references**
(`ref`), the concrete id *values* are arbitrary — gold may call someone `4`
while pred calls the same someone `94`. So before comparing references, the
aligner must discover a **bijection** (a one-to-one mapping) between gold ids
and pred ids, *per scope*.

When the entities carry distinguishing attributes the bijection falls out of
ordinary property matching. 1-WL is the machinery that rescues the cases where
attributes are **not** enough and only an entity's **position in the reference
graph** can tell it apart from a look-alike.


## 1. What is a "definer"? (the hazy term, pinned down)

The word shows up all over the referential code (`definer_array_path`,
`definer_schema_path`, "definer item"). Here is the whole idea in plain terms.

An **id scope** is a named pool of entities — e.g. all the *people*, or all the
*tracks*. Three layered meanings of "definer":

| Term | What it is | In our example |
|------|------------|----------------|
| **definer field** | the `idScope` primitive that *names* (defines) an entity's id | `people[*].id`, `tracks[*].id` |
| **definer item** | one array element that *introduces* one entity (it carries the definer field) | one `{ "id": 4, "name": "Dave", … }` object |
| **definer array** | the array whose items are the definer items — the alignable *list* of entities | the `people` array, the `tracks` array |

So a **definer** is the place an id is *born*. Every other appearance of that id
value is a **reference** (`ref`) — a *pointer back* to a definer item, not a new
definition.

Why must the definer field live **inside an array**? Because the entities form a
*list we can align*: deriving the bijection is literally "match the gold definer
items to the pred definer items." The id field on its own is a leaf; the array
around it is what makes "person #1 in gold ↔ which person in pred?" a
well-posed assignment problem.

The code stores two paths per scope:

- **`definer_array_path`** — schema path to the *definer array* (`people`).
- **`definer_schema_path`** — schema path to the *definer field* itself
  (`people` → `items` → `id`).

The slice between them (`definer_schema_path[len(definer_array_path):]`) is the
"how do I get from one definer item to its id" suffix. We will see all of this
printed in §5.

**One-line mantra:** *a definer item creates an entity; a `ref` just mentions
one.*


## 2. The example: a research group with people and tracks

Two scopes:

- **`track`** — research tracks (`"AI"`, `"Systems"`). Tracks have distinct
  names, so their bijection is easy. *This scope is resolved first* (you'll see
  why in §6).
- **`person`** — group members. **This is the scope WL has to work for.**

The wiring:

- each person has a `track` reference (a person belongs to one track),
- `mentorships` are **directed, typed edges** between people, each tagged with a
  `track` (this gives WL edges *labels*, including a cross-scope one),
- `committees` have a `members` list — a **k-ary** relation that becomes a
  *hub* in the graph.

The trap we set: there are **two people who are byte-identical by their own
fields** — both `{"name": "Dave", "role": "Engineer", "track": Systems}`. Their
properties cannot tell them apart. Only the graph can.


In [1]:
from object_aligner import ObjectAligner

schema = {
    "type": "object",
    "properties": {
        "tracks": {                                  # id scope "track"
            "type": "array", "order": "align",
            "items": {"type": "object", "properties": {
                "id":   {"type": "integer", "idScope": "track"},
                "name": {"type": "string"},
            }},
        },
        "people": {                                  # id scope "person"
            "type": "array", "order": "align",
            "items": {"type": "object", "properties": {
                "id":    {"type": "integer", "idScope": "person"},
                "name":  {"type": "string"},
                "role":  {"type": "string"},
                "track": {"type": "integer", "ref": "track"},   # ref into the OTHER scope
            }},
        },
        "mentorships": {                             # directed, typed edges among people
            "type": "array", "order": "align",
            "items": {"type": "object", "properties": {
                "mentor": {"type": "integer", "ref": "person"},
                "mentee": {"type": "integer", "ref": "person"},
                "track":  {"type": "integer", "ref": "track"},  # cross-scope edge label
            }},
        },
        "committees": {                              # k-ary membership -> hub vertices
            "type": "array", "order": "align",
            "items": {"type": "object", "properties": {
                "name":    {"type": "string"},
                "members": {"type": "array", "order": "align",
                            "items": {"type": "integer", "ref": "person"}},
            }},
        },
    },
}


def build(pid, tid, people_order, tracks_order):
    # Construct one side. pid/tid map logical names -> concrete id values.
    obj = {
        "tracks": [
            {"id": tid["AI"],  "name": "AI"},
            {"id": tid["Sys"], "name": "Systems"},
        ],
        "people": [
            {"id": pid["alice"], "name": "Alice", "role": "Manager",  "track": tid["AI"]},
            {"id": pid["bob"],   "name": "Bob",   "role": "Engineer", "track": tid["AI"]},
            {"id": pid["carol"], "name": "Carol", "role": "Engineer", "track": tid["Sys"]},
            {"id": pid["dave1"], "name": "Dave",  "role": "Engineer", "track": tid["Sys"]},  # twin
            {"id": pid["dave2"], "name": "Dave",  "role": "Engineer", "track": tid["Sys"]},  # twin
        ],
        # Both Daves are mentees, tagged with the SAME track, differing only in
        # WHO mentors them (Alice vs Carol) -> the tie can only break after the
        # mentors' own colors diverge (round 2).
        "mentorships": [
            {"mentor": pid["alice"], "mentee": pid["dave1"], "track": tid["AI"]},
            {"mentor": pid["carol"], "mentee": pid["dave2"], "track": tid["AI"]},
        ],
        # Both Daves sit in the SAME committee (symmetric there, so the hub
        # cannot separate them either).
        "committees": [
            {"name": "Hiring", "members": [pid["alice"], pid["dave1"], pid["dave2"]]},
            {"name": "Social", "members": [pid["bob"], pid["carol"]]},
        ],
    }
    obj["people"] = [obj["people"][i] for i in people_order]
    obj["tracks"] = [obj["tracks"][i] for i in tracks_order]
    return obj


gold = build(
    pid={"alice": 1, "bob": 2, "carol": 3, "dave1": 4, "dave2": 5},
    tid={"AI": 100, "Sys": 200},
    people_order=(0, 1, 2, 3, 4), tracks_order=(0, 1),
)
# pred: different id values AND a scrambled list order, to prove WL recovers the
# mapping from structure, not from position.
pred = build(
    pid={"alice": 91, "bob": 92, "carol": 93, "dave1": 94, "dave2": 95},
    tid={"AI": 71, "Sys": 72},
    people_order=(4, 1, 3, 0, 2), tracks_order=(1, 0),
)

aligner = ObjectAligner(schema)                         # id_disambiguation="wl" (default)
print("score with WL (default):", aligner.metric(gold, pred))
print("score with WL disabled :", ObjectAligner(schema, id_disambiguation="none").metric(gold, pred))


score with WL (default): {'score': 1.0}
score with WL disabled : {'score': 0.9166666666666666}


**Read the gap.** WL recovers a perfect `1.0`. With WL off, the aligner
can only tie-break the two identical Daves *arbitrarily*; it picks the wrong one
and the score drops to `0.9166…`. The deficit is exactly the two mentorship /
committee references that now point at the wrong Dave. Small but real — and the
rest of the notebook shows precisely how WL earns the missing `1/12`.


In [2]:
gold

{'tracks': [{'id': 100, 'name': 'AI'}, {'id': 200, 'name': 'Systems'}],
 'people': [{'id': 1, 'name': 'Alice', 'role': 'Manager', 'track': 100},
  {'id': 2, 'name': 'Bob', 'role': 'Engineer', 'track': 100},
  {'id': 3, 'name': 'Carol', 'role': 'Engineer', 'track': 200},
  {'id': 4, 'name': 'Dave', 'role': 'Engineer', 'track': 200},
  {'id': 5, 'name': 'Dave', 'role': 'Engineer', 'track': 200}],
 'mentorships': [{'mentor': 1, 'mentee': 4, 'track': 100},
  {'mentor': 3, 'mentee': 5, 'track': 100}],
 'committees': [{'name': 'Hiring', 'members': [1, 4, 5]},
  {'name': 'Social', 'members': [2, 3]}]}

In [3]:
pred

{'tracks': [{'id': 72, 'name': 'Systems'}, {'id': 71, 'name': 'AI'}],
 'people': [{'id': 95, 'name': 'Dave', 'role': 'Engineer', 'track': 72},
  {'id': 92, 'name': 'Bob', 'role': 'Engineer', 'track': 71},
  {'id': 94, 'name': 'Dave', 'role': 'Engineer', 'track': 72},
  {'id': 91, 'name': 'Alice', 'role': 'Manager', 'track': 71},
  {'id': 93, 'name': 'Carol', 'role': 'Engineer', 'track': 72}],
 'mentorships': [{'mentor': 91, 'mentee': 94, 'track': 71},
  {'mentor': 93, 'mentee': 95, 'track': 71}],
 'committees': [{'name': 'Hiring', 'members': [91, 94, 95]},
  {'name': 'Social', 'members': [92, 93]}]}

## 3. The call chain (two scopes now)

Same flow as the first notebook, but `_derive_id_mappings` loops over **two**
scopes in topological order, and the earlier-resolved scope (`track`) feeds the
later one (`person`).

```
align() / metric()                                       [object_aligner.py]
└─ _align_with_ctx()
   ├─ _validate_referential(gold)         gold id sets    [_aligner_referential.py]
   ├─ _collect_pred_ids(pred)             pred id sets     [_aligner_referential.py]
   ├─ _derive_id_mappings()               loops scopes in topo order
   │    for scope in (track, person):
   │      _derive_single_scope()
   │        ├─ _align_helper(...)         property cost matrix   [_aligner_core.py]
   │        ├─ _build_ref_graph()         data -> abstract graph [_aligner_wl.py]
   │        ├─ wl_tokens()                graph -> per-side colors [_wl.py]
   │        ├─ _apply_wl()                fold colors into cost   [_aligner_wl.py]
   │        └─ linear_sum_assignment()    Hungarian -> mapping    [scipy]
   └─ _align_helper(gold, pred, ...)      normal scoring; refs use the mappings
```


## 4. `_collect_id_scopes` — finding the definers and refs

Runs once at construction. It walks the schema, records every `idScope` as a
scope with its definer paths, and attaches every `ref` site to the scope it
points at. We now have **two** scopes; let's print what the aligner discovered,
using the definer vocabulary from §1.


In [4]:
for name, scope in aligner._id_scopes.items():
    print(f"=== scope {name!r} ===")
    print("  definer_array_path :", scope.definer_array_path)
    print("  definer_schema_path:", scope.definer_schema_path)
    suffix = scope.definer_schema_path[len(scope.definer_array_path):]
    print("  item -> id suffix  :", suffix, " (how to get an id out of one definer item)")
    print("  ref_paths          :")
    for rp in scope.ref_paths:
        print("      ", rp)
    print()


=== scope 'track' ===
  definer_array_path : (('properties', 'tracks'), ('items',))
  definer_schema_path: (('properties', 'tracks'), ('items',), ('properties', 'id'))
  item -> id suffix  : (('properties', 'id'),)  (how to get an id out of one definer item)
  ref_paths          :
       (('properties', 'people'), ('items',), ('properties', 'track'))
       (('properties', 'mentorships'), ('items',), ('properties', 'track'))

=== scope 'person' ===
  definer_array_path : (('properties', 'people'), ('items',))
  definer_schema_path: (('properties', 'people'), ('items',), ('properties', 'id'))
  item -> id suffix  : (('properties', 'id'),)  (how to get an id out of one definer item)
  ref_paths          :
       (('properties', 'mentorships'), ('items',), ('properties', 'mentor'))
       (('properties', 'mentorships'), ('items',), ('properties', 'mentee'))
       (('properties', 'committees'), ('items',), ('properties', 'members'), ('items',))



Read this against §1. For `person`:

- the **definer array** is `people`,
- the **definer field** is `people → items → id`,
- and the references to a person come from three places: `mentorships.mentor`,
  `mentorships.mentee`, and `committees.members`.

For `track`, notice one of its `ref_paths` is `people → items → track` — a
reference that lives **inside the `person` definer items**. That nesting is
exactly what creates the dependency we resolve next.


## 5. `_toposort_scopes` — which scope to solve first

Rule: if a reference to scope **B** appears *inside* scope **A**'s definer
subtree, then **B must be solved first** (so A can use B's finished mapping as
evidence). `people[*].track` is a `track` reference living inside a `person`
definer item, so **`track` must be resolved before `person`**.


In [5]:
print("resolution order:", aligner._scope_order)

# Show the dependency that forces it: a `track` ref nested under person's definer array.
person = aligner._id_scopes["person"]
track  = aligner._id_scopes["track"]
for rp in track.ref_paths:
    nested = len(rp) > len(person.definer_array_path) and rp[:len(person.definer_array_path)] == person.definer_array_path
    print(f"  track ref at {rp}  nested under person definers? {nested}")


resolution order: ('track', 'person')
  track ref at (('properties', 'people'), ('items',), ('properties', 'track'))  nested under person definers? True
  track ref at (('properties', 'mentorships'), ('items',), ('properties', 'track'))  nested under person definers? False


So `track` is solved first. When we later solve `person`, the `track`
mapping is already known — and it shows up in *two* places: the `person` cost
matrix (a person's `track` ref contributes real signal) **and** the WL edge
labels (the cross-scope fold in §9c).


## 6. `_walk_data` — schema path → concrete values

A schema path is a *recipe*; `_walk_data(data, path)` follows it and yields
every `(value, data_path)` reached. `("items",)` fans out over a whole list.


In [6]:
print("people definer items (gold):")
for value, data_path in aligner._walk_data(gold, person.definer_array_path):
    print("   at", data_path, "->", value)

print("\nevery person-id referenced by mentorships.mentor (gold):")
mentor_path = next(rp for rp in person.ref_paths if rp[-1] == ("properties", "mentor"))
print("  ", [v for v, _ in aligner._walk_data(gold, mentor_path)])


people definer items (gold):
   at ('people', 0) -> {'id': 1, 'name': 'Alice', 'role': 'Manager', 'track': 100}
   at ('people', 1) -> {'id': 2, 'name': 'Bob', 'role': 'Engineer', 'track': 100}
   at ('people', 2) -> {'id': 3, 'name': 'Carol', 'role': 'Engineer', 'track': 200}
   at ('people', 3) -> {'id': 4, 'name': 'Dave', 'role': 'Engineer', 'track': 200}
   at ('people', 4) -> {'id': 5, 'name': 'Dave', 'role': 'Engineer', 'track': 200}

every person-id referenced by mentorships.mentor (gold):
   [1, 3]


## 7. The id sets — `_validate_referential` and `_collect_pred_ids`

`_validate_referential(gold)` is strict (duplicate or dangling gold ids raise);
`_collect_pred_ids(pred)` is tolerant. We build a `ctx` exactly like `align()`
does and populate both, for **both** scopes.


In [7]:
from object_aligner._matchtypes import _AlignContext

ctx = _AlignContext()
ctx.gold_ids = aligner._validate_referential(gold)
ctx.pred_ids = aligner._collect_pred_ids(pred)
print("gold ids:", ctx.gold_ids)
print("pred ids:", ctx.pred_ids)


gold ids: {'track': {200, 100}, 'person': {1, 2, 3, 4, 5}}
pred ids: {'track': {72, 71}, 'person': {91, 92, 93, 94, 95}}


## 8. Resolving `track` first, then the `person` cost matrix

`_derive_id_mappings` walks the scopes in topological order. Let's run it and
look at both mappings. `track` is solved by plain property matching (distinct
names); its result is stored in `ctx.current_mappings` and becomes evidence for
`person`.


In [8]:
mappings, pred_excess = aligner._derive_id_mappings(gold, pred, ctx)
print("track mapping :", mappings["track"])    # solved first, by names
print("person mapping:", mappings["person"])   # solved second, with WL help


track mapping : {100: 71, 200: 72}
person mapping: {1: 91, 2: 92, 3: 93, 4: 94, 5: 95}


Now the interesting one: the **`person` property cost matrix**. For each
gold person *i* and pred person *j*, `_derive_single_scope` scores their
*ordinary properties* via `_align_helper`, with one twist — refs **into the
scope being solved** (`person`) are masked to `1.0` (no bootstrapping). But
refs into the *already-resolved* `track` scope are **not** masked: they
contribute real signal. Let's rebuild it by hand.


In [9]:
import numpy as np

# Mask self-scope (person) refs; keep already-resolved scopes (track) as evidence.
ctx.current_mappings = {k: v for k, v in mappings.items() if k != "person"}
ctx.mask_scope = "person"
item_schema = aligner._get_schema_node(aligner.schema, person.definer_array_path)

gold_items = list(aligner._walk_data(gold, person.definer_array_path))
pred_items = list(aligner._walk_data(pred, person.definer_array_path))
n, m = len(gold_items), len(pred_items)

suffix = person.definer_schema_path[len(person.definer_array_path):]
def extract_id(item):
    for val, _ in aligner._walk_data(item, suffix):
        return val
gold_id_list = [extract_id(it) for it, _ in gold_items]
pred_id_list = [extract_id(it) for it, _ in pred_items]

d = max(n, m)
cost = np.zeros((d, d))
for i in range(n):
    for j in range(m):
        cost[i][j] = aligner._align_helper(gold_items[i][0], pred_items[j][0], item_schema, ctx)["match"].score

np.set_printoptions(precision=3, suppress=True)
print("gold people ids (rows):", gold_id_list)
print("pred people ids (cols):", pred_id_list)
print("property cost matrix:")
print(cost)


gold people ids (rows): [1, 2, 3, 4, 5]
pred people ids (cols): [95, 92, 94, 91, 93]
property cost matrix:
[[0.543 0.673 0.543 1.    0.423]
 [0.5   1.    0.5   0.673 0.5  ]
 [0.871 0.5   0.871 0.423 1.   ]
 [1.    0.5   1.    0.543 0.871]
 [1.    0.5   1.    0.543 0.871]]


**This matrix is genuinely mixed** — unlike the all-`1.0` matrix in the
first notebook. Most people are separated by their `name` / `role` / resolved
`track`. But find the two **Dave** rows (gold ids `4` and `5`): they are
**identical**. Property matching cannot choose which pred-Dave each gold-Dave
maps to — a `1.0`-vs-`1.0` tie. That single tie is the entire job left for WL.


In [10]:
# Make the residual tie explicit: which gold rows are identical?
for i in range(n):
    for k in range(i + 1, n):
        if np.allclose(cost[i, :m], cost[k, :m]):
            print(f"gold ids {gold_id_list[i]} and {gold_id_list[k]} have IDENTICAL cost rows -> tie")


gold ids 4 and 5 have IDENTICAL cost rows -> tie


## 9. `_build_ref_graph` — data → an abstract labeled graph

WL works on a plain **directed, labeled graph**, not on JSON. `_build_ref_graph`
is the translator. Compared with the first notebook, this graph is much richer:
it has **hub vertices** and **non-empty labels**.


In [11]:
gold_graph = aligner._build_ref_graph(gold, person, ctx, is_gold=True)
pred_graph = aligner._build_ref_graph(pred, person, ctx, is_gold=False)

print("GOLD vertices:", dict(gold_graph.vertices))
print("GOLD incidences:")
for e in gold_graph.incidences:
    print(f"    {e.src!s:>14} -> {e.dst!s:<14} role={e.role}  label={e.label}")


GOLD vertices: {1: (), 2: (), 3: (), 4: (), 5: (), ('__hub__', 0): (), ('__hub__', 1): ()}
GOLD incidences:
                 4 -> 1              role=('edge', (('properties', 'mentee'),), (('properties', 'mentor'),))  label=(('xref', 'track', "(('properties', 'track'),)", 71),)
                 5 -> 3              role=('edge', (('properties', 'mentee'),), (('properties', 'mentor'),))  label=(('xref', 'track', "(('properties', 'track'),)", 71),)
                 1 -> ('__hub__', 0) role=('member', (('properties', 'members'), ('items',)))  label=(('str', 'name', 'Hiring'),)
                 4 -> ('__hub__', 0) role=('member', (('properties', 'members'), ('items',)))  label=(('str', 'name', 'Hiring'),)
                 5 -> ('__hub__', 0) role=('member', (('properties', 'members'), ('items',)))  label=(('str', 'name', 'Hiring'),)
                 2 -> ('__hub__', 1) role=('member', (('properties', 'members'), ('items',)))  label=(('str', 'name', 'Social'),)
                 3 -> ('__hub_

Two things to notice that the simple notebook never showed:

1. **Hub vertices** `('__hub__', 0)` and `('__hub__', 1)` appear alongside the
   real person ids — one per committee (more on this in §9b).
2. **Labels are non-empty.** Mentorship edges carry an `('xref', 'track', …)`
   label (a *cross-scope fold*); committee incidences carry
   `('str', 'name', 'Hiring' / 'Social')`. These labels are what stop WL from
   matching structurally-different relations.

The next three subsections explain *how* those vertices, edges, and labels were
produced.


### 9a. `_carrier_path` — what object *owns* each reference

A reference sits inside some object — its **carrier**. `_carrier_path` finds the
smallest enclosing object-that-is-an-array-item for a ref site. `mentor` and
`mentee` share the *mentorship* item as carrier (so they fuse into one directed
edge); `members[*]` is carried by the *committee* item (so co-members form one
group relation).


In [12]:
for rp in person.ref_paths:
    print(f"ref {str(rp[-1]):<28} carrier = {aligner._carrier_path(rp)}")


ref ('properties', 'mentor')     carrier = (('properties', 'mentorships'), ('items',))
ref ('properties', 'mentee')     carrier = (('properties', 'mentorships'), ('items',))
ref ('items',)                   carrier = (('properties', 'committees'), ('items',))


`mentor` and `mentee` report the **same** carrier (the `mentorships[*]`
item) → they will become a single directed edge. `committees[*].members`
reports the *committee* item → all its members attach to one shared structure.


### 9b. `_emit_incidences` — turning a carrier's endpoints into graph edges

Given the endpoints a carrier collects, `_emit_incidences` picks one of three
shapes — and this trichotomy is exactly what keeps the encoding inside 1-WL's
expressive power:

| # endpoints | distinct roles | becomes | example here |
|---|---|---|---|
| 1 | — | a **unary self-tag** (`src == dst`) | (none in this data) |
| 2 | 2 | a **directed edge** `src → dst` | a `mentorship` (`mentor`/`mentee`) |
| otherwise | — | a **star to a fresh hub vertex** | a `committee` (`members: […]`) |

- The **directed-edge** case produced our `mentee → mentor` edges. (The role
  tuple records which ref was which, so direction survives; here the endpoints
  are sorted, putting `mentee` first.)
- The **hub** case is why each committee created a `('__hub__', k)` vertex: a
  3-member committee can't be a single 2-endpoint edge, so all members star into
  one hub. This makes membership **order-invariant** — swapping two members
  doesn't change the graph.

So the *number and roles* of a carrier's endpoints decide the shape, with no
hand-tuning.


#### Why three shapes — and what about ternary / quaternary relations?

A fair question: the directed edge is just an *ordered pair*, so why not a native
*triple* `(a, b, c)` or *quadruple* `(a, b, c, d)`? The constraint comes from
1-WL itself. 1-WL only knows how to pass messages along **pairwise, directed,
labeled edges** between vertices and aggregate them into a multiset (that is the
update rule in §10). It has **no native notion of a hyperedge** joining three or
more vertices at once. So *every* relation — whatever its arity — has to be
rendered as pairwise edges over the definer vertices. The three shapes are simply
the three faithful ways to do that:

- **Unary (1 participant)** → a labeled **self-loop** on that vertex. There is no
  second endpoint to draw an edge to, so the relation becomes a *tag* attached to
  the vertex (`role=("unary", …)`).
- **Binary with two *distinct* roles (2 participants)** → one **directed edge**
  `a → b`. This is the tightest, most WL-expressive encoding of a binary relation:
  each end directly sees the other's color one hop away, and the `(dir, role)`
  pair records the asymmetry (who is the `mentor`, who the `mentee`). The pair is
  stored in a canonical order (sorted by role) with *both* roles kept in the role
  tuple, so direction is never ambiguous.
- **Everything else** → a **star to a fresh hub vertex**: a synthetic
  "relation vertex" is created and *every* participant gets an edge to it
  (`role=("member", <its role>)`). "Everything else" means **arity ≥ 3** *and*
  **the symmetric binary case** (two participants that share the *same* role, e.g.
  two siblings — an unordered pair, so it must **not** become a directed edge).

So, directly answering the questions:

- **Can we represent a triple `(a, b, c)` or a quadruple?** Yes — that is exactly
  what the hub does, for *any* arity $k$. A $k$-ary relation becomes $k$ edges, all
  pointing at one shared hub.
- **Does order/position survive for $k \ge 3$?** It survives **through the roles**,
  not through tuple position. If the three participants have *distinct* roles
  (`giver`, `recipient`, `item`), each carries its own role on its edge to the hub,
  so the positions are fully recoverable. If they share a role (a homogeneous
  `members: […]` list) the relation is **order-invariant** by construction —
  permuting the members produces the identical star, which is the correct
  semantics for a *set* of members.
- **So does the hub ignore order?** It ignores order *only among participants that
  share a role* (which is the membership-list case). Participants with different
  roles remain distinguishable. The hub encodes a **multiset of `(role, member)`
  incidences**, not a sequence.

Two natural follow-ups:

- **Why not always use a hub, even for binary edges?** A hub adds an extra vertex,
  so a distinction now has to travel *participant → hub → participant*, costing one
  extra refinement round to propagate. Keeping unary tags and binary directed edges
  in their native form makes them maximally discriminative and is the standard
  graph encoding; the hub indirection is reserved for relations that genuinely
  cannot be a single pairwise edge.
- **Why a hub rather than a clique (connect all pairs)?** A fresh hub *per relation
  instance* preserves the **grouping** — "these are the members of *this one*
  committee." A clique would forget the grouping: two overlapping committees would
  blur into one tangle of pairwise edges, and "jointly in one relation" could no
  longer be told from "pairwise related." The hub keeps each relation a separate,
  identifiable object.

The cell below calls `_emit_incidences` directly on synthetic endpoint lists so
you can see each shape (and verify the order claims):


In [17]:
from object_aligner._wl import RefGraph
emit = ObjectAligner._emit_incidences   # static; takes (endpoints, label, graph, hub_counter)

def show_shape(title, endpoints):
    g = RefGraph(); hub_counter = [0]
    emit(endpoints, label=("<carrier-label>",), graph=g, hub_counter=hub_counter)
    print(f"{title}")
    print(f"   endpoints = {endpoints}")
    extra = [v for v in g.vertices]
    print(f"   hub vertices created: {extra if extra else 'none'}")
    for e in g.incidences:
        print(f"      {e.src!s:>4} -> {e.dst!s:<14} role={e.role}")
    print()

show_shape("unary  (1 ref)            -> self-loop tag", [("tag", "A")])
show_shape("binary, DISTINCT roles    -> directed edge", [("source", "A"), ("target", "B")])
show_shape("binary, SAME role         -> hub (symmetric pair)", [("member", "A"), ("member", "B")])
show_shape("ternary, DISTINCT roles   -> hub, roles keep position", [("giver", "A"), ("recipient", "B"), ("item", "C")])
show_shape("ternary, SAME role        -> hub, order-invariant", [("member", "A"), ("member", "B"), ("member", "C")])

# Order-invariance check: shuffling a same-role members list yields identical incidences.
g1 = RefGraph(); g2 = RefGraph()
emit([("m", "A"), ("m", "B"), ("m", "C")], ("L",), g1, [0])
emit([("m", "C"), ("m", "A"), ("m", "B")], ("L",), g2, [0])
set1 = {(e.src, e.dst, e.role) for e in g1.incidences}
set2 = {(e.src, e.dst, e.role) for e in g2.incidences}
print("same-role members: reordering gives identical incidence set?", set1 == set2)


unary  (1 ref)            -> self-loop tag
   endpoints = [('tag', 'A')]
   hub vertices created: none
         A -> A              role=('unary', 'tag')

binary, DISTINCT roles    -> directed edge
   endpoints = [('source', 'A'), ('target', 'B')]
   hub vertices created: none
         A -> B              role=('edge', 'source', 'target')

binary, SAME role         -> hub (symmetric pair)
   endpoints = [('member', 'A'), ('member', 'B')]
   hub vertices created: [('__hub__', 0)]
         A -> ('__hub__', 0) role=('member', 'member')
         B -> ('__hub__', 0) role=('member', 'member')

ternary, DISTINCT roles   -> hub, roles keep position
   endpoints = [('giver', 'A'), ('recipient', 'B'), ('item', 'C')]
   hub vertices created: [('__hub__', 0)]
         A -> ('__hub__', 0) role=('member', 'giver')
         B -> ('__hub__', 0) role=('member', 'recipient')
         C -> ('__hub__', 0) role=('member', 'item')

ternary, SAME role        -> hub, order-invariant
   endpoints = [('member',

### 9c. `_carrier_label` / `_exact_scalars` — baking hard evidence onto edges

A relation's **label** is a sorted tuple of two kinds of evidence about its
carrier:

1. the carrier's own **exactly-comparable scalars** (`_exact_scalars`: strings,
   ints, bools, enums — but **not** floats, which would split identical edges
   over rounding noise, and **not** id/ref fields). A committee's `name` is such
   a scalar.
2. any **references the carrier makes to an already-resolved *higher* scope**,
   mapped into pred space on the gold side so both sides agree. A mentorship's
   `track` ref is such a cross-scope fold.

Let's call it directly on one mentorship and one committee carrier:


In [18]:
# A mentorship carrier (Alice -> Dave1, in track AI=100):
mentorship0 = gold["mentorships"][0]
m_carrier = aligner._carrier_path(person.ref_paths[0])      # the mentorships[*] item
m_label = aligner._carrier_label(mentorship0, m_carrier, person, ctx, is_gold=True)
print("mentorship0:", mentorship0)
print("  exact scalars:", aligner._exact_scalars(mentorship0, aligner._get_schema_node(aligner.schema, m_carrier)))
print("  carrier label:", m_label, " <- track 100 folded as resolved pred id", ctx.current_mappings['track'][100])

# A committee carrier (name 'Hiring'):
committee0 = gold["committees"][0]
members_ref = next(rp for rp in person.ref_paths if rp[-1] == ("items",))
c_carrier = aligner._carrier_path(members_ref)              # the committees[*] item
c_label = aligner._carrier_label(committee0, c_carrier, person, ctx, is_gold=True)
print("\ncommittee0:", committee0["name"])
print("  carrier label:", c_label, " <- the committee name, an exact scalar")


mentorship0: {'mentor': 1, 'mentee': 4, 'track': 100}
  exact scalars: ()
  carrier label: (('xref', 'track', "(('properties', 'track'),)", 71),)  <- track 100 folded as resolved pred id 71

committee0: Hiring
  carrier label: (('str', 'name', 'Hiring'),)  <- the committee name, an exact scalar


**Why the cross-scope fold matters.** The label `('xref', 'track', …, 71)`
uses the *pred* track id `71` even though we evaluated the gold side — because
the gold track `100` was already mapped to pred `71` in §8. So a gold mentorship
"in the AI track" and a pred mentorship "in the AI track" get the **same** label
on both sides, without ever consulting the `person` mapping we are still trying
to compute. That is what keeps WL free of any chicken-and-egg bootstrapping.


## 10. `wl_tokens` — the color refinement, step by step

This is the heart of the method, so we take it slowly. `wl_tokens` gives every
vertex an integer **color** such that two vertices share a color **iff 1-WL
cannot tell them apart** — and, crucially, the colors are **comparable across
the gold and pred graphs**, which is what lets us read a bijection off "same
color".

We build up to it in stages:

- **§10a** — *why* the two graphs are refined together over a "disjoint union"
  (with a tiny standalone example),
- **§10b** — the whole algorithm as pseudocode,
- **§10c / §10d / §10e** — the three concrete stages (adjacency → relabel →
  rounds), each in its own short cell,
- **§10f** — the update rule **dissected**: the exact `dir`, `role`, `label`,
  `color` values for the two Daves, round by round.

First, the answer the library produces — keep it in view as the target:


In [15]:
from object_aligner._wl import wl_tokens

gold_tokens, pred_tokens = wl_tokens(gold_graph, pred_graph, mode="tie_break")
print("gold tokens:", gold_tokens)
print("pred tokens:", pred_tokens)


gold tokens: {1: 0, 2: 6, 3: 1, 4: 4, 5: 5, ('__hub__', 0): 2, ('__hub__', 1): 3}
pred tokens: {95: 5, 92: 6, 94: 4, 91: 0, 93: 1, ('__hub__', 0): 2, ('__hub__', 1): 3}


Every person got a distinct color, and the colors line up across sides
(`4 ↔ 94`, `5 ↔ 95`, …). The rest of §10 explains how those integers are
produced and why they are safe to compare across the two graphs.


### 10a. Why a *disjoint union*? (the motivation)

Here is the problem we must avoid. Suppose we refined the **gold** graph on its
own: we would hand out colors `0, 1, 2, …` to gold vertices. Then we refined the
**pred** graph on its own: again `0, 1, 2, …`. Now gold's "color 1" and pred's
"color 1" are **just two independent counters** — there is no reason the same
number means the same structural role on both sides. We could not compare them.

We can't fix this by first matching gold vertices to pred vertices either —
*finding that matching is the whole point*; assuming it would be circular
(the bootstrapping trap).

**The fix is the disjoint union.** Put *all* vertices from *both* graphs into one
shared bag (we tag each vertex with its side, `(0, id)` for gold and `(1, id)`
for pred, so they never collide — there are **no edges between the two graphs**,
hence *disjoint*). Then, each round, we compute every vertex's structural
**signature** and convert signatures to integers with **one shared dictionary**
spanning the whole bag. Consequence: if a gold vertex and a pred vertex have the
*identical* signature, they are forced to receive the *identical* integer — for
free, without ever pairing them up first.

A minimal example makes it concrete. Two separate one-edge graphs, `A → B` and
`X → Y`, with no shared names:


In [16]:
from object_aligner._wl import RefGraph, _RefEdge

#   gold:  A -> B            pred:  X -> Y      (same shape, disjoint names)
gold_mini = RefGraph(
    vertices={"A": (), "B": ()},
    incidences=[_RefEdge(src="A", dst="B", role=("edge", "src", "dst"), label=())],
)
pred_mini = RefGraph(
    vertices={"X": (), "Y": ()},
    incidences=[_RefEdge(src="X", dst="Y", role=("edge", "src", "dst"), label=())],
)

g_mini, p_mini = wl_tokens(gold_mini, pred_mini, mode="tie_break")
print("gold:", g_mini)
print("pred:", p_mini)
print("A ↔ X (both the 'source' end)?", g_mini["A"] == p_mini["X"])
print("B ↔ Y (both the 'target' end)?", g_mini["B"] == p_mini["Y"])
print("A ≠ B (source distinguished from target)?", g_mini["A"] != g_mini["B"])


gold: {'A': 1, 'B': 0}
pred: {'X': 1, 'Y': 0}
A ↔ X (both the 'source' end)? True
B ↔ Y (both the 'target' end)? True
A ≠ B (source distinguished from target)? True


`A` (a vertex with one *outgoing* edge and no incoming one) and `X` (same
description) land on the **same** color; `B` and `Y` (one *incoming*, no
outgoing) share the **other** color. The shared relabeling dictionary made gold
and pred colors mean the same thing — and nothing ever told the algorithm that
`A` "is" `X`. That is the entire trick, scaled up to our research-group graph
below.


### 10b. The algorithm in pseudocode

```text
INPUT:  gold_graph, pred_graph         # each = vertices + directed, labeled incidences
OUTPUT: an integer color per vertex, comparable across the two graphs

bag = {(0, v) : v in gold} ∪ {(1, v) : v in pred}      # disjoint union; tag by side

# adjacency[x] lists, for vertex x, one tuple per incident edge:
#   (dir, role, label, neighbor)      dir ∈ {"out", "in"}
build adjacency by scanning every edge of both graphs

color[x] = 0  for all x in bag                          # round 0: one shared start color

repeat:
    for x in bag:
        nbrs = sorted( (dir, role, label, color[u])
                       for each incident (dir, role, label, u) of x )   # a multiset
        signature[x] = (color[x], nbrs)                 # old color + who I'm next to
    color = relabel(signature)                          # ONE shared signature→int map
until the number of distinct colors stops increasing    # stable partition

return color restricted to gold,  color restricted to pred
```

Three things to internalise:

- the **starting color is the same constant for everyone** (under `tie_break`),
  so round 0 carries *no* information — all structure is discovered by the loop;
- a vertex's new color depends on its neighbors' **previous-round** colors, so
  information spreads **one hop per round**;
- `relabel` is the shared dictionary from §10a — it is what couples the two
  sides.

The next three cells are exactly these pieces, run on the real graph.


### 10c. Stage 1 — build the disjoint-union adjacency

Tag every vertex by side (`0` = gold, `1` = pred) and record, for each edge, an
`("out", …)` entry on its source and an `("in", …)` entry on its destination.
Each entry is the 4-tuple `(dir, role, label, neighbor)` the update rule will
consume.


In [17]:
graphs = {0: gold_graph, 1: pred_graph}

adjacency = {}
for side, graph in graphs.items():
    for vid in graph.vertices:
        adjacency[(side, vid)] = []

for side, graph in graphs.items():
    for e in graph.incidences:
        if (side, e.src) in adjacency:
            adjacency[(side, e.src)].append(("out", e.role, e.label, (side, e.dst)))
        if (side, e.dst) in adjacency:
            adjacency[(side, e.dst)].append(("in", e.role, e.label, (side, e.src)))

keys = list(adjacency)
print("vertices in the shared bag:", len(keys), "(5 gold + 5 pred people + 2 hubs each side)")
print("\nadjacency of gold Dave1 = vertex (0, 4):")
for entry in adjacency[(0, 4)]:
    print("   ", entry)


vertices in the shared bag: 14 (5 gold + 5 pred people + 2 hubs each side)

adjacency of gold Dave1 = vertex (0, 4):
    ('out', ('edge', (('properties', 'mentee'),), (('properties', 'mentor'),)), (('xref', 'track', "(('properties', 'track'),)", 71),), (0, 1))
    ('out', ('member', (('properties', 'members'), ('items',))), (('str', 'name', 'Hiring'),), (0, ('__hub__', 0)))


Dave1 has two entries: an `"out"` edge to his mentor (Alice) and an
`"out"` membership into the "Hiring" hub. We will read these exact tuples apart
in §10f.


### 10d. Stage 2 — the shared `relabel` step

This is the coupling from §10a, in code. Given a `signature` for every vertex in
the bag, it collects the **distinct** signatures, orders them by `repr` (a
deterministic order independent of hashing/insertion), numbers them `0, 1, 2, …`,
and hands each vertex the number of its signature. Equal signatures — on *either*
side — therefore collapse to the *same* integer.


In [18]:
def relabel(signatures, ks):
    distinct = sorted({signatures[k] for k in ks}, key=repr)
    token = {sig: i for i, sig in enumerate(distinct)}
    return {k: token[signatures[k]] for k in ks}

# Tiny demo: p and q share a signature, r differs.
demo = {"p": ("same",), "q": ("same",), "r": ("other",)}
print(relabel(demo, list(demo)), " <- p and q collapse to one integer; r gets another")


{'p': 1, 'q': 1, 'r': 0}  <- p and q collapse to one integer; r gets another


### 10e. Stage 3 — run the rounds

Now the loop from the pseudocode. We seed one constant color, then repeatedly
rebuild signatures and `relabel`, stopping when the partition stops growing. We
**store every round's colors** in `history` so §10f can dissect them.


In [19]:
def show(color, title):
    people = {vid: color[(0, vid)] for vid in gold_graph.vertices if not isinstance(vid, tuple)}
    nparts = len({color[k] for k in keys})
    tie = "Daves TIED" if color[(0, 4)] == color[(0, 5)] else "Daves SPLIT"
    print(f"{title:8} {nparts} colors   gold people={people}   <- {tie}")

history = []
color = relabel({k: ("c0", ()) for k in keys}, keys)   # round 0: one constant color
history.append(color)
show(color, "round 0")

prev = len({color[k] for k in keys})
for r in range(1, len(keys) + 1):
    signatures = {}
    for k in keys:
        nbrs = sorted(
            ((dr, role, label, color[n]) for (dr, role, label, n) in adjacency[k]),
            key=repr,
        )
        signatures[k] = (color[k], tuple(nbrs))
    color = relabel(signatures, keys)
    history.append(color)
    show(color, f"round {r}")
    parts = len({color[k] for k in keys})
    if parts == prev:
        print("         -> partition stable; stop.")
        break
    prev = parts


round 0  1 colors   gold people={1: 0, 2: 0, 3: 0, 4: 0, 5: 0}   <- Daves TIED
round 1  6 colors   gold people={1: 0, 2: 5, 3: 1, 4: 4, 5: 4}   <- Daves TIED
round 2  7 colors   gold people={1: 0, 2: 6, 3: 1, 4: 4, 5: 5}   <- Daves SPLIT
round 3  7 colors   gold people={1: 0, 2: 6, 3: 1, 4: 4, 5: 5}   <- Daves SPLIT
         -> partition stable; stop.


### 10f. The update rule, dissected

The rule again, then its four ingredients, with the **exact values** for our
graph:

$$c_{t+1}(v) = \mathrm{relabel}\Big(c_t(v),\ \{\!\!\{\,(\text{dir},\,\text{role},\,\text{label},\,c_t(u)) : u \in N(v)\,\}\!\!\}\Big)$$

For each neighbor $u$ of $v$, the update collects a 4-tuple:

- **`dir`** — `"out"` or `"in"`: which way the directed edge points relative to
  $v$. (Being someone's mentor is different from being their mentee.)
- **`role`** — *which relation*, and $v$'s slot in it. For a mentorship it is
  `('edge', <mentee-path>, <mentor-path>)`; for a committee it is
  `('member', <members-path>)`. (These are raw schema-path tuples — verbose, but
  that verbosity is what keeps two different relations from ever colliding.)
- **`label`** — the carrier's hard evidence from §9c: a committee `name`
  (`('str','name','Hiring')`) or a folded cross-scope reference
  (`('xref','track',…,71)`).
- **`color`** — the neighbor's color **from the previous round**. This is the
  recursive ingredient: it is how a distinction *two hops away* reaches $v$ on
  the *next* round.

Let's print these for Dave1, then compare Dave1 vs Dave2 round by round.


In [20]:
name = {1: "Alice", 2: "Bob", 3: "Carol", 4: "Dave1", 5: "Dave2"}
def vname(k):
    side, v = k
    return f"hub{v[1]}" if isinstance(v, tuple) else name[v]
def leaf(path):                       # (('properties','mentee'),) -> 'mentee'
    return path[-1][-1] if path and isinstance(path[-1], tuple) else path
def gloss_role(role):
    if role[0] == "edge":   return f"edge {leaf(role[1])}→{leaf(role[2])}"
    if role[0] == "member": return "committee membership"
    if role[0] == "unary":  return f"unary {leaf(role[1])}"
    return str(role)
def gloss_label(label):
    out = []
    for t in label:
        if t[0] == "xref":  out.append(f"{t[1]}→pred#{t[-1]}")
        elif t[0] == "str": out.append(f"{t[1]}={t[2]!r}")
        else:               out.append(str(t))
    return "{" + ", ".join(out) + "}"

print("Dave1 = gold vertex (0, 4). Its two incidence 4-tuples (dir, role, label, neighbor):\n")
for (dr, role, label, nbk) in sorted(adjacency[(0, 4)], key=repr):
    print(f"  dir   = {dr!r}")
    print(f"  role  = {role!r}")
    print(f"           i.e. {gloss_role(role)}")
    print(f"  label = {label!r}")
    print(f"           i.e. {gloss_label(label)}")
    print(f"  neighbor = {vname(nbk)}   (its color entering round 1 was {history[0][nbk]})\n")


Dave1 = gold vertex (0, 4). Its two incidence 4-tuples (dir, role, label, neighbor):

  dir   = 'out'
  role  = ('edge', (('properties', 'mentee'),), (('properties', 'mentor'),))
           i.e. edge mentee→mentor
  label = (('xref', 'track', "(('properties', 'track'),)", 71),)
           i.e. {track→pred#71}
  neighbor = Alice   (its color entering round 1 was 0)

  dir   = 'out'
  role  = ('member', (('properties', 'members'), ('items',)))
           i.e. committee membership
  label = (('str', 'name', 'Hiring'),)
           i.e. {name='Hiring'}
  neighbor = hub0   (its color entering round 1 was 0)



Now the comparison that explains the two-round split. A helper rebuilds a
vertex's signature *entering* a given round (using the previous round's colors),
exactly as Stage 3 does:


In [21]:
def signature_entering(round_idx, k):
    prev = history[round_idx - 1]
    nbrs = sorted(((dr, role, label, prev[n]) for (dr, role, label, n) in adjacency[k]), key=repr)
    return (prev[k], tuple(nbrs))

print("ROUND 1  (neighbor colors = round-0 colors, which are all 0)")
s4_1, s5_1 = signature_entering(1, (0, 4)), signature_entering(1, (0, 5))
print("  Dave1 signature:", s4_1)
print("  Dave2 signature:", s5_1)
print("  -> identical:", s4_1 == s5_1, " => Daves stay TIED after round 1\n")

print("ROUND 2  (neighbor colors = round-1 colors)")
print(f"  Dave1's mentor Alice has round-1 color {history[1][(0, 1)]};"
      f"  Dave2's mentor Carol has round-1 color {history[1][(0, 3)]}")
s4_2, s5_2 = signature_entering(2, (0, 4)), signature_entering(2, (0, 5))
print("  Dave1 signature:", s4_2)
print("  Dave2 signature:", s5_2)
print("  -> identical:", s4_2 == s5_2, " => the differing mentor color SPLITS the Daves")


ROUND 1  (neighbor colors = round-0 colors, which are all 0)
  Dave1 signature: (0, (('out', ('edge', (('properties', 'mentee'),), (('properties', 'mentor'),)), (('xref', 'track', "(('properties', 'track'),)", 71),), 0), ('out', ('member', (('properties', 'members'), ('items',))), (('str', 'name', 'Hiring'),), 0)))
  Dave2 signature: (0, (('out', ('edge', (('properties', 'mentee'),), (('properties', 'mentor'),)), (('xref', 'track', "(('properties', 'track'),)", 71),), 0), ('out', ('member', (('properties', 'members'), ('items',))), (('str', 'name', 'Hiring'),), 0)))
  -> identical: True  => Daves stay TIED after round 1

ROUND 2  (neighbor colors = round-1 colors)
  Dave1's mentor Alice has round-1 color 0;  Dave2's mentor Carol has round-1 color 1
  Dave1 signature: (4, (('out', ('edge', (('properties', 'mentee'),), (('properties', 'mentor'),)), (('xref', 'track', "(('properties', 'track'),)", 71),), 0), ('out', ('member', (('properties', 'members'), ('items',))), (('str', 'name', 'Hi

**This is the whole mechanism in one screen.**

- In round 1 every neighbor still has color `0`, so the two Daves — same role
  (`mentee`), same edge label (track AI), same committee hub — produce
  **byte-identical signatures** and stay tied.
- The two mentorship edges point at *different* people (Alice vs Carol), and
  those two got **different colors in round 1** (`0` vs `1`). So in round 2 the
  Daves' signatures differ in exactly one slot — the mentor's color — and the
  shared `relabel` therefore assigns them **different** integers.

Because gold and pred were refined in the same bag, the split happens identically
on both sides, and matching colors hand us `4 ↔ 94`, `5 ↔ 95`.


**The two-round payoff, restated:** information **propagates one hop per
round**. The fact that distinguishes the twins (*who their mentor is*) lives one
hop away, and that mentor's own distinguishing fact reaches the twin only on the
*second* round. 1-WL is exactly this hop-by-hop spreading of structural
information, run jointly over both graphs so the resulting colors are comparable.


## 11. `_apply_wl` — folding colors back into the cost

Turn "same color" into a nudge. First an **agreement matrix** `w` with
`w[i][j] = 1` iff gold person *i* and pred person *j* share a WL color. Then the
default `"tie_break"` adds a *tiny* `eps · w` — small enough that any pair
already separated by properties keeps its ranking, so structure decides **only**
where properties were tied (our two Daves).


In [22]:
w = np.zeros((d, d))
for i in range(n):
    gtok = gold_tokens.get(gold_id_list[i])
    for j in range(m):
        if gtok is not None and pred_tokens.get(pred_id_list[j]) == gtok:
            w[i][j] = 1.0

print("WL agreement matrix w (1 = same color):")
print(w)

score_matrix = aligner._apply_wl(cost, w, n, m)
print("\nscore matrix after tie-break (cost + eps*w):")
print(score_matrix)
print("\nthe two Dave rows are no longer identical:",
      not np.allclose(score_matrix[3, :m], score_matrix[4, :m]))


WL agreement matrix w (1 = same color):
[[0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]]

score matrix after tie-break (cost + eps*w):
[[0.543 0.673 0.543 1.002 0.423]
 [0.5   1.002 0.5   0.673 0.5  ]
 [0.871 0.5   0.871 0.423 1.002]
 [1.    0.5   1.002 0.543 0.871]
 [1.002 0.5   1.    0.543 0.871]]

the two Dave rows are no longer identical: True


The numbers barely move (`eps` is deliberately sub-gap), but the Dave
rows — previously identical — now each have a unique best column. The ambiguity
is gone exactly where it existed and nowhere else.


## 12. The Hungarian step — read off the bijection

In [23]:
from scipy.optimize import linear_sum_assignment

row_ind, col_ind = linear_sum_assignment(-score_matrix)
mapping = {}
matched = set()
for ri, ci in zip(row_ind, col_ind):
    if ri < n and ci < m:
        g_id, p_id = gold_id_list[ri], pred_id_list[ci]
        if p_id is not None and p_id not in matched:
            mapping[g_id] = p_id; matched.add(p_id)
        else:
            mapping[g_id] = None
print("derived person bijection (gold id -> pred id):", mapping)


derived person bijection (gold id -> pred id): {1: 91, 2: 92, 3: 93, 4: 94, 5: 95}


## 13. Cross-check against the public path

Everything above re-implemented the internals by hand. Confirm it matches what
`ObjectAligner` actually computes end-to-end:


In [24]:
match, real_ctx = aligner._align_with_ctx(gold, pred)
print("track  mapping (library):", real_ctx.current_mappings["track"])
print("person mapping (library):", real_ctx.current_mappings["person"])
print("final score:", aligner.metric(gold, pred))


track  mapping (library): {100: 71, 200: 72}
person mapping (library): {1: 91, 2: 92, 3: 93, 4: 94, 5: 95}
final score: {'score': 1.0}


With both bijections in hand, normal scoring resumes: a `ref` field is
scored by looking the gold id up in the mapping and checking pred points at the
mapped id. Every mentorship and committee reference now lands on the right Dave,
so the score is a perfect `1.0`.


## 14. When WL *can't* help — and shouldn't

If the two Daves were wired **symmetrically** — same mentor, same committees,
same everything — then swapping them would be a true graph **automorphism** and
*no* algorithm could prefer one pairing. WL correctly leaves that residual
ambiguity alone (and `warn_on_ambiguous_mapping=True` would fire on it). The
first notebook (`explain_ra_wl.ipynb`, §12) walks through such a case, along
with the known 1-WL blind spot (one 6-cycle vs two 3-cycles). The takeaway: WL
breaks every tie that *structure* can break, and honestly reports the ones it
can't.


## 15. Recap

1. **Definer** = where an id is born: a **definer array** (`people`) of
   **definer items**, each carrying the **definer field** (`idScope`). A `ref`
   only *mentions* a definer item by id.
2. **`_collect_id_scopes`** records the definer paths and ref sites for each scope.
3. **`_toposort_scopes`** orders scopes so a referenced scope (`track`) resolves
   before the scope that references it (`person`).
4. **`_derive_single_scope`** builds a property **cost matrix** with self-scope
   refs masked; here it cleanly separates everyone *except* the two property-twin
   Daves.
5. **`_build_ref_graph`** translates data into a labeled graph — **directed
   edges** (mentorships) and **hub stars** (committees), with **labels** carrying
   exact scalars and a **cross-scope fold** of the resolved `track`.
6. **`wl_tokens`** refines gold and pred **jointly**; the twins stay tied through
   round 1 and split in round 2 once their mentors' colors diverge.
7. **`_apply_wl`** nudges the cost by a sub-gap `eps` only where it was tied.
8. **`linear_sum_assignment`** reads off the bijection; refs then score through it.

The one idea, again: **when an entity's own fields can't distinguish it from a
twin, its position in the reference graph usually can — and 1-WL turns "position
in the graph" into a color that is comparable across gold and pred.**
